# BERTopic for topic modelling

Welcome back!
<br>

<img src="https://raw.githubusercontent.com/MaartenGr/BERTopic/master/images/logo.png" width="40%">

# RECAP

In the previous session, we worked with a dataset of tweets collected from the Office of the United Nations High Commissioner for Human Rights (2017–2022). The aim was not to test predefined categories, but to explore what themes emerge from the data using topic modelling.

We began by setting up the environment and downloading the dataset. Because the original file contained a formatting issue around a specific row, we imported it in two parts and reconstructed a clean version of the dataset. This step highlighted a practical point: real-world data often requires manual inspection and adjustment before analysis.

We then focused on text preprocessing. Rather than applying a single cleaning step, we constructed a pipeline of functions to:

remove emojis,
strip URLs and mentions,
clean hashtags,
remove noisy characters,
and standardise spacing.

This allowed us to move from raw social media text to a more consistent representation suitable for modelling.

After preprocessing, we filtered the dataset to English-language tweets and created a cleaned text column. These cleaned texts formed the input to our topic model.

We then introduced BERTopic and fitted the model using:

a sentence transformer (all-mpnet-base-v2) to generate embeddings,
UMAP to reduce dimensionality,
and a keyphrase-based vectorizer to improve topic representation.

This pipeline allowed us to move from individual tweets to clusters of semantically similar texts.

Finally, we explored the outputs of the model by:

inspecting topic frequencies,
examining top terms within topics,
and looking at representative documents.

At this stage, the focus was on understanding what the model produces and how topics are constructed.

# This Week...
In this session, we move from generating topics to working more closely with the model outputs.

We focus on how to interpret topics in a more systematic way, going beyond simple inspection of top words. This includes examining topic composition, comparing topics, and identifying patterns in how documents are grouped.

We also make use of visualisations to better understand the structure of the model and how topics relate to one another.

Finally, we consider ways to improve topic quality in practice. This includes adjusting model components and discussing extensions, such as adapting embedding models to better fit specific datasets.

In [ ]:
import subprocess
import sys
from importlib.metadata import distributions

packages = [
    "bertopic",
    "hdbscan",
    "keyphrase_vectorizers",
    "nbformat",
    "emoji"
]

installed = {dist.metadata["Name"].lower() for dist in distributions()}
missing = [pkg for pkg in packages if pkg.lower() not in installed]

print(missing)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])


In [ ]:
import locale
import pandas as pd
import re
import emoji
import string
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from keyphrase_vectorizers import KeyphraseCountVectorizer
locale.getpreferredencoding = lambda: "UTF-8"

# Reload Data and Model
We briefly rerun the first session so the model and its outputs are available for inspection.

In [ ]:
# Reload the data
!curl -O https://raw.githubusercontent.com/world-politics-datalab/un_hum_rights_office_tweets/main/un_office_humrights_tweets_sept4_2017_sept3_2022.csv

# Preprocess the data
data = pd.read_csv("un_office_humrights_tweets_sept4_2017_sept3_2022.csv", header=0, nrows=4037, encoding='utf-8',  quotechar='"')
data = data.iloc[:,:88]

!tail -n 17130 un_office_humrights_tweets_sept4_2017_sept3_2022.csv > temp.csv

data2 = pd.read_csv("temp.csv", encoding='utf-8',  quotechar='"', header=None)
data2.drop(data2.columns[[14, 15]], axis=1, inplace=True)

data2 = pd.DataFrame(data=data2.values, columns=data.columns)

# data_all = data.append(data2,ignore_index=True) # version for use with older versions of pandas

df = pd.concat((data, data2), ignore_index=True)
df.to_csv("un_tweets_corrected.csv", encoding='utf-8')

def strip_emoji(text):
    """
    Remove all emoji characters from the text.
    Using the emoji package to safely filter emojis.
    """
    return emoji.replace_emoji(text, replace="")

def strip_all_entities(text):
    """
    Clean links, mentions, ASCII noise, punctuation,
    and convert the text to lowercase.
    """
    # Remove newline characters and lowercase text
    text = text.replace("\r", "").replace("\n", " ").lower()

    # Remove mentions (@username) and URLs
    text = re.sub(r"(?:\@|https?\://)\S+", "", text)

    # Remove non-ASCII characters (e.g., stray corrupted bytes)
    text = re.sub(r"[^\x00-\x7f]", r"", text)

    # Remove punctuation and special characters
    banned_list = string.punctuation + "Ã" + "±" + "ã" + "¼" + "â" + "»" + "§"
    table = str.maketrans("", "", banned_list)
    text = text.translate(table)

    return text

def clean_hashtags(tweet):
    """
    Remove trailing hashtags entirely,
    but keep words that have '#' inside the sentence by removing the '#'.
    """
    # Remove hashtags at the end of the sentence
    new_tweet = " ".join(
        word.strip()
        for word in re.split(
            r"#(?!(?:hashtag)\b)[\w-]+(?=(?:\s+#[\w-]+)*\s*$)", tweet
        )
    )
    # Remove the '#' symbol for inline hashtags
    new_tweet2 = " ".join(word.strip() for word in re.split("#|_", new_tweet))
    return new_tweet2

def filter_chars(text):
    """
    Remove words containing '$' or '&' characters.
    These symbols often appear in corrupted text or ads.
    """
    cleaned = []
    for word in text.split(" "):
        if ("$" in word) or ("&" in word):
            cleaned.append("")  # remove word
        else:
            cleaned.append(word)
    return " ".join(cleaned)

def remove_mult_spaces(text):
    """
    Replace multiple spaces with a single space.
    Ensures clean and consistent spacing.
    """
    return re.sub(r"\s\s+", " ", text)

    # select only English data
en_data = df.loc[df['lang'] == "en"]

clean_texts = []
for t in en_data["text"]:
    cleaned = strip_emoji(t)                 # 1. remove emojis
    cleaned = strip_all_entities(cleaned)    # 2. remove URLs, mentions, punctuation
    cleaned = clean_hashtags(cleaned)        # 3. clean hashtag patterns
    cleaned = filter_chars(cleaned)          # 4. remove unwanted symbols ($, &)
    cleaned = remove_mult_spaces(cleaned)    # 5. collapse multiple spaces
    clean_texts.append(cleaned)

en_data["text_clean"] = clean_texts
docs = list(en_data["text_clean"])


# Training the Model

# 01 - Text representation (create embeddings)
# See: https://huggingface.co/sentence-transformers
embedding_model = "all-mpnet-base-v2"

# 02 - Reduce dimensionality (UMAP)
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

# 03 - Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

# 04 - Topic representation
vectorizer_model = KeyphraseCountVectorizer(max_df = int(0.5*len(docs)),min_df = 5, decay = 0.1, delete_min_df=5.0)

# 05 - Create topic
#ctfidf_model = ClassTfidfTransformer()

# 06 - (Optional) Fine-tune topic representations with
# a `bertopic.representation` model
#representation_model = KeyBERTInspired()


# NOTE: earlier versions of KeyphraseCountVectorizer do not use the `decay` and `delete_min_df` params
try: # works with KeyphraseCountVectorizer v0.13
    topic_model = BERTopic(embedding_model="all-mpnet-base-v2",
                           umap_model = umap_model,
                           hdbscan_model= hdbscan_model,
                           vectorizer_model= vectorizer_model,
                           language="english",
                           calculate_probabilities=True,
                           verbose=True,
                        )
except TypeError: # in case of older version of KeyphraseCountVectorizer
    topic_model = BERTopic(embedding_model="all-mpnet-base-v2",
                           umap_model = umap_model,
                           hdbscan_model= hdbscan_model,
                           vectorizer_model=KeyphraseCountVectorizer(max_df = int(0.5*len(docs)), min_df = 5),
                           language="english",
                           calculate_probabilities=True,
                           verbose=True,
                        )

# =============================
# Fit the data on the model
# =============================

topics, probs = topic_model.fit_transform(docs)


# Interpreting Topics
Now we are back at where we left last week.
Let's recheck how topics are distributed. We can start by looking at which topics are most frequent and how much of the data they capture.

In [ ]:
# Overview of topics
topic_model.get_topic_info()

In [ ]:
# Inspect a specific topic (keywords)
topic_model.get_topic(1)

In [ ]:
# Inspect representative documents
topic_model.representative_docs_

# Comparing Topics

We can compare topics qualitatively by inspecting the keywords in each

In [ ]:
topic_model.get_topic(3)

In [ ]:
topic_model.get_topic(7)

In [ ]:
topic_model.get_representative_docs(3)

In [ ]:
topic_model.get_representative_docs(7)

## Concept ~ Topic
We can quantitatively inspect topics that have the closest similarity to a concept using BERTopic's built in similarity.


In [ ]:
# Below is an example of how we can find topics that are closer to the concept of 'torture'.
similar_topics, similarity = topic_model.find_topics("torture", top_n=5)
similar_topics, similarity


In [ ]:
for topic in similar_topics:
    print(topic_model.get_topic(topic))

Similarity scores can be interpreted as:
closer to 1-> very similar
closer to 0-> weak similarity

## Topic vs Topic
We can compare topics using a heatmap.

In [ ]:
topic_model.visualize_heatmap()

We can also compare topics quantitatively by looking at their c-TF-IDF representations. Each topic is defined by a weighted set of words, and cosine similarity allows us to measure how similar these word distributions are.

This provides a more direct comparison of topics based on their defining terms, rather than relying only on visual inspection or broader semantic embeddings.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Get c-TF-IDF matrix
ctfidf = topic_model.c_tf_idf_

# Compare two topics
topic_a = 4
topic_b = 9

similarity = cosine_similarity( #Do these two topics rely on similar words, and with similar importance?
    ctfidf[topic_a],
    ctfidf[topic_b]
)

print(similarity)


# Visualisation

## Visualise Structure: Topic Map

In [ ]:
topic_model.visualize_topics()

## Visualise Representation :  Term Frequencies

In [ ]:
topic_model.visualize_barchart()

## Visualise Topics : Wordcloud

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Select a topic
topic_id = 20

# Get words and weights
topic_words = dict(topic_model.get_topic(topic_id))

# Create word cloud
wc = WordCloud(background_color="white")
wc.generate_from_frequencies(topic_words)

# Plot
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title(f"Topic {topic_id}")
plt.show()

## Visualize Topic Hierarchy
The topics that were created can be hierarchically reduced. In order to understand the potential hierarchical structure of the topics, we can use scipy.cluster.hierarchy to create clusters and visualize how they relate to one another. This might help selecting an appropriate nr_topics when reducing the number of topics that you have created.

In [ ]:
topic_model.visualize_hierarchy()

# Improving BERTopic

We can improve our topic model in several ways. These include reducing the number of topics, refining how topics are represented, and incorporating additional information such as annotated data. Before making any changes, however, we first need to identify where the model may be underperforming.

Potential issues to look for include topics that overlap in meaning, topics that are too broad or too specific, and topics that are difficult to interpret. We may also observe that some topics contain noisy or loosely related documents, suggesting that the clustering is not fully coherent. Identifying these issues allows us to make more targeted adjustments, rather than modifying the model arbitrarily.

## Topic Reduction

One common issue is that the model may generate too many topics, some of which capture very similar or overlapping content. This can make the results harder to interpret, as related themes are split across multiple topics rather than being grouped together. In such cases, topic reduction can be used to merge similar topics into broader clusters. This helps simplify the structure of the model, making it easier to interpret, although the resulting topics may still require further inspection.

To decide whether we should reduce the topics, we can inspect the heatmap.
If multiple topics are highliy similar, this suggests redundancy and indicates that topic reduction may be appropriate

In [ ]:
topic_model.visualize_heatmap()

Topic reduction modifies the model in place. To compare the model before and after reduction, we need to create a copy of the original model before applying the reduction step.

In [ ]:
import copy
original_model = copy.deepcopy(topic_model)


In [ ]:
# Reducing Topics

# Reduce the number of topics
reduced_model = topic_model.reduce_topics(docs, nr_topics=30)

# Inspect the new topics
reduced_model.get_topic_info()

In [ ]:
len(reduced_model.get_topic_info())

In [ ]:
len(original_model.get_topic_info())

#Comparing Original and Reduced Model

We can inpect side-by-side topic tables

In [ ]:
orig_topics = original_model.get_topic_info()
red_topics  = reduced_model.get_topic_info()

print(orig_topics.head(10))
print(red_topics.head(10))

We can also inspect the mapping between the original and reduced models to identify which topics have been merged. This will help us compare specific topics

In [ ]:
reduced_model.topic_mapper_.get_mappings()

In [ ]:
from IPython.display import display

print("Original topics (before reduction):")
print("Topic 0:", original_model.get_topic(0))
print("Topic 3:", original_model.get_topic(3))
print("Topic 4:", original_model.get_topic(4))

print("\nReduced topic:")
print("Topic 12:", reduced_model.get_topic(12))

In [ ]:
original_model.visualize_topics()

In [ ]:
reduced_model.visualize_topics()

## Fine Tuning
Another way to improve our results is to fine-tune our model. Fine-tuning refers to adjusting the topic model to improve the quality and interpretability of the topics it produces. Rather than rebuilding the model from scratch, we refine its outputs by modifying how topics are represented or by incorporating additional information. This allows us to better align the model with the structure of the data or with specific research goals.


In practice, fine-tuning can take several forms. For example, we may adjust how topics are grouped, refine their representations, or incorporate annotated data to guide the model towards more meaningful distinctions. These steps allow us to move from a purely unsupervised model to one that reflects more domain-specific knowledge.

### Guided Topic Modelling
One way to fine-tune a topic model is to guide it using domain knowledge. This can be done by specifying sets of seed words that represent themes we expect to find in the data. The model then uses these seed words to steer the formation of topics, resulting in more meaningful and interpretable clusters.

In [ ]:
# Define Seed Topics:
seed_topic_list = [
    ["syria", "conflict", "civilians"],
    ["covid-19", "pandemic", "public health"],
    ["gender equality", "women", "rights"],
    ["climate change", "environment", "emissions"],
    ["torture", "detention", "ill-treatment"]
]

In [ ]:
# Create Model with Seeds
topic_model_seeded = BERTopic(
    embedding_model="all-mpnet-base-v2",
    seed_topic_list=seed_topic_list,
    calculate_probabilities=True,
    verbose=True
)

In [ ]:
# Fit Model (do not run without a GPU)
topics, probs = topic_model_seeded.fit_transform(docs)

In [ ]:
print("Original:", len(original_model.get_topic_info()))
print("Reduced:", len(reduced_model.get_topic_info()))
print("Fine-tuned:", len(topic_model_seeded.get_topic_info()))

In [ ]:
topic_model_seeded.visualize_topics()

Reduction controls the number of topics.
Fine-tuning controls the meaning of topics.

## **Tasks**

**1) Concept-based comparison**

Use find_topics() to explore how each model represents a concept (e.g. "torture", "climate", "covid").

Do the returned topics relate to the concept?
Are they more relevant in the fine-tuned model compared to the original and reduced models?

**2) Compare topic representations**

Select 2–3 topics and examine their top words across models.

Are the words more coherent in the fine-tuned model?
Do topics appear more specific or meaningful?

**3) Examine noise (Topic -1)**

Compare Topic -1 across the three models.

Does the amount of noise change?
Does fine-tuning reduce or redistribute noise?

**4) Compare visualisations**

Use different visualisation methods (e.g. visualize_topics(), visualize_heatmap(), visualize_barchart()).

Do you observe clearer groupings in any model?
Are topics more or less clustered?
**Reminder** Make sure you use original_model for the initial model, as topic_model was updated during topic reduction.

## Other Ways to Refine Topics

###Improving topic representation
Using representation models (e.g. KeyBERT-inspired) to surface more meaningful words

###Adjusting the vectorizer
Removing very common or generic words (e.g. “photo”, “thank”)

###Changing embeddings or clustering parameters
Affecting how topics are formed and separated

# The End